In [0]:
spark.sql("USE CATALOG e_comm")
spark.sql("USE SCHEMA silver")

In [0]:
spark.sql("create table if not exists silver.orders(order_id string, customer_id string, order_status string, order_purchase_timestamp string, order_approved_at string, order_delivered_carrier_date string, order_delivered_customer_date string, order_estimated_delivery_date string) using delta ")

In [0]:
from pyspark.sql.functions import *
import requests
from io import StringIO
import pandas as pd

df = spark.read.table("e_comm.bronze.orders")
stored_ts = spark.sql("select ts from e_comm.bronze.metadata_table where tname = 'orders'").collect()[0][0]
df = df.filter(col('order_purchase_timestamp') > stored_ts)
print(stored_ts)
max_ts = df.select(max("order_purchase_timestamp")).collect()[0][0]
print(max_ts)
if max_ts != None:
    spark.sql(f"update e_comm.bronze.metadata_table set ts = '{max_ts}' where tname = 'orders'")
    df = df.withColumn("merge_flag", lit(False))
    df.write.format("delta").mode("append").option("mergeSchema", "True").saveAsTable("e_comm.silver.orders")
    print("reached")




In [0]:
%sql

select count(*) from e_comm.silver.orders 